# Teardown

**WARNING: this permanently deletes every `pamap2-har` AWS resource** (S3 data lake incl. all versions, Feature Group, Glue/Athena, SageMaker models/packages/endpoints, Model Card, CloudWatch alarms/dashboard, project log streams). IAM (`LabRole`) and `config/project_config.json` are never touched.

Default is a dry run. To delete: set `DRY_RUN=False` and `CONFIRM` to the bucket name printed below. `KEEP_DATA=True` keeps the bucket and `raw/` + `curated/` objects.

In [ ]:
DRY_RUN = True
CONFIRM = ""
KEEP_DATA = False

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import boto3, pandas as pd
from src.pamap2.config import load_config
from src.pamap2.teardown import teardown, discover, plan_order

cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
clients = {n: session.client(n) for n in ("s3", "sagemaker", "glue", "athena", "cloudwatch", "logs")}
print("Bucket (use as CONFIRM):", cfg["bucket"])

In [ ]:
errs = []
plan = plan_order(discover(clients, cfg, KEEP_DATA, errs))
print(pd.DataFrame([{"kind": r.kind, "name": r.name} for r in plan]).to_string(index=False))
if errs:
    print("\nDISCOVERY FAILURES:")
    print(pd.DataFrame([{"kind": e.kind, "error": e.extra["error"]} for e in errs]).to_string(index=False))

In [ ]:
rows = teardown(clients, cfg, dry_run=DRY_RUN, confirm=CONFIRM, keep_data=KEEP_DATA)
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(summary["status"].value_counts().to_string())
failed = summary[summary["status"] == "failed"]
if len(failed):
    raise RuntimeError(f"{len(failed)} teardown step(s) failed - see summary above")